In [ ]:
##this script is used for extracting the ms2 data from the targetd msms data and used for spectrum comparison.

In [10]:
#helper functions
# !pip install pyopenms
# !pip install numpy
# !pip install tabulate
# !pip install matchms
import re
import numpy as np
import matchms 
from matchms import calculate_scores
from matchms import Spectrum
from matchms.similarity import CosineGreedy
import pandas as pd
import pyopenms as oms
import tabulate
import os 
import copy
import warnings
import logging
# Suppress specific warnings
warnings.filterwarnings("ignore", message=".*")

def ppm_error(mz, target):
    return abs(mz - target) / target * 1e6

def preprocess_spectra(filepath, target_mz, ce_level, tolerance=5):
    # Extract spectra with specified collision energy level for target m/z 
    if target_mz is None:
        raise ValueError("Please provide a target m/z value.")
    ce_level = float(ce_level)
    tolerance = float(tolerance)
    target_mz = float(target_mz)
    spectra = oms.MSExperiment()
    oms.MzMLFile().load(filepath, spectra)
    ms2_spec = spectra.getSpectra()

    ms2_query = oms.MSExperiment()
    for s in ms2_spec:
        if s.getMSLevel() == 1:
            continue
        precursors = s.getPrecursors()
        mslevel = s.getMSLevel()
        collision_energy = precursors[0].getMetaValue("collision energy")
        ms_error = ppm_error(precursors[0].getMZ(), target_mz)
        if ms_error < tolerance and mslevel == 2 and collision_energy == ce_level:
            ms2_query.addSpectrum(s)
        else :
            continue

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]

    # print(f'Number of MS2 spectra before merge: {len(ms2beforemerg)}')

    if len(ms2beforemerg) == 0:
        # print(f"No spectra found with specified parameters for {ce_level}.")
        return None
    
    elif len(ms2beforemerg) > 1:
        #merget the ms spectra
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("mz_tolerance", 1e-3)
        param.setValue("rt_tolerance", "5.0")
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        return ms2_query
    elif len(ms2beforemerg) == 1:
        return ms2_query
    


def parse_msp_file(msp_file_path, polarity):
    """
    Parses the .msp file to extract all spectrum information.
    
    Args:
        msp_file_path (str): Path to the .msp file.

    Returns:
        dict: A dictionary where keys are InChIKey or SMILES and values are lists of spectra data.
    """
    spectra = {}
    if polarity == "positive":
        p_type = "[M+H]+"
    elif polarity == "negative":
        p_type = "[M-H]-"
    else:
        raise ValueError("Invalid polarity. Please specify 'positive' or 'negative'.")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            key = None

            for line in lines:
                if line.startswith("InChIKey:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("SMILES:") and key is None:
                    key = line.split(": ")[1].strip()
                elif line.startswith("Spectrum_type:"):
                    spectrum_type = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"):
                    precursor_type = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    spectrum_data = lines[lines.index(line)+1:lines.index(line)+1+num_peaks]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra with "Spectrum_type: MS2"
            if spectrum_type == "MS2" and precursor_type == p_type  and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data)]
                })

    return spectra


def parse_mona_database(db_file_path, polarity):
    """
    Parses an alternate database format to extract all spectrum information based on polarity.

    Args:
        db_file_path (str): Path to the database file.
        polarity (str): The ion polarity to filter ("positive" or "negative").

    Returns:
        dict: A dictionary where keys are InChIKey or SMILES and values are lists of spectra data.
    """
    spectra = {}

    if polarity == "positive":
        p_type = "[M+H]+"
    elif polarity == "negative":
        p_type = "[M-H]-"
    else:
        raise ValueError("Invalid polarity. Please specify 'positive' or 'negative'.")

    with open(db_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('NAME: ')  # "NAME:" marks the beginning of each spectrum.
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            key = None

            for line in lines:
                if line.startswith("INCHIKEY:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("SMILES:") and key is None:
                    key = line.split(": ")[1].strip()
                elif line.startswith("PRECURSORTYPE:"):
                    ionization = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    spectrum_data = [tuple(map(float, peak.split())) for peak in lines[lines.index(line)+1:] if peak.strip()]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra matching the desired polarity
            if ionization == p_type and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": spectrum_data
                })

    return spectra


def get_spectrum_by_key(spectra, key):
    """
    Retrieves all spectrum information by InChIKey or SMILES.

    Args:
        spectra (dict): Parsed spectra data.
        key (str): The InChIKey or SMILES to search for.

    Returns:
        dict: A dictionary where keys are collision energies and values are spectra.
    """
    if key not in spectra:
        return {}

    grouped_spectra = {}
    for spec in spectra[key]:
        collision_energy = spec["metadata"].get("Collision_energy", spec["metadata"].get("COLLISIONENERGY", "Unknown"))
        if collision_energy not in grouped_spectra:
            grouped_spectra[collision_energy] = []
        grouped_spectra[collision_energy].append(spec["spectrum"])

    return grouped_spectra

def normalize_and_filter(spectrum, baseline):
    """
    Accepts:
      - pyopenms/matchms Spectrum with .get_peaks()
      - list of (mz, intensity) tuples
      - tuple: (mz_array, intensity_array)
    Returns:
      - (mz_res, int_res): top-N peaks (sorted by m/z)
    """
    # -------- 1. unify input format: get mz and intensity as np.arrays --------
    if hasattr(spectrum, "get_peaks"):
        # pyOpenMS or matchms Spectrum
        mz, intensity = spectrum.get_peaks()
        mz = np.asarray(mz, dtype=float)
        intensity = np.asarray(intensity, dtype=float)

    elif isinstance(spectrum, list) and len(spectrum) > 0 and isinstance(spectrum[0], (tuple, list)):
        # list of (mz, intensity) pairs
        mz, intensity = zip(*spectrum)
        mz = np.asarray(mz, dtype=float)
        intensity = np.asarray(intensity, dtype=float)

    elif isinstance(spectrum, tuple) and len(spectrum) == 2:
        # (mz_array, intensity_array)
        mz = np.asarray(spectrum[0], dtype=float)
        intensity = np.asarray(spectrum[1], dtype=float)

    else:
        raise ValueError("Spectrum format is not recognized. "
                         "Expected object with .get_peaks(), list of (mz,intensity), or (mz_array,int_array) tuple.")

    # -------- 2. select top 8 most intense peaks --------
    if len(intensity) > 8:
        top_indices = np.argsort(intensity)[-8:]
    else:
        top_indices = np.argsort(intensity)

    mz_res = mz[top_indices]
    int_res = intensity[top_indices]

    # -------- 3. sanity check & sort by m/z --------
    if len(mz_res) != len(int_res):
        raise ValueError("The length of mz and intensity arrays do not match after filtering.")

    sorted_indices = np.argsort(mz_res)
    mz_res = mz_res[sorted_indices]
    int_res = int_res[sorted_indices]

    return mz_res, int_res

# def normalize_and_filter(spectrum, baseline):
#     try:
#         mz, intensity = spectrum.get_peaks()
#     except AttributeError: 
#         if isinstance(spectrum, list) and all(isinstance(i, tuple) for i in spectrum):
#             mz, intensity = zip(*spectrum)
#             mz = np.array(mz)
#             intensity = np.array(intensity)
#         else:
#             raise ValueError("Spectrum format is not recognized.")
#     except:
#         if isinstance(spectrum, tuple) and len(spectrum) == 2:
#             mz = np.array(spectrum[0])
#             intensity = np.array(spectrum[1])
#         else:
#             raise ValueError("Spectrum format is not recognized.")
    
#     # select the top 8 most intense peaks
#     if len(intensity) > 8:
#         top_indices = np.argsort(intensity)[-8:]
#         mz_res = mz[top_indices]
#         int_res = intensity[top_indices]
#     else:
#         top_indices = np.argsort(intensity)
#         mz_res = mz[top_indices]
#         int_res = intensity[top_indices]

#     # Assert that the length of mz and intensity are the same
#     if len(mz_res) != len(int_res):
#         raise ValueError("The length of mz and intensity arrays do not match after filtering.")
    
#     # Sort mz values in ascending order and reorder intensities correspondingly.
#     sorted_indices = np.argsort(mz_res)
#     mz_res = mz_res[sorted_indices]
#     int_res = int_res[sorted_indices]
#     return mz_res, int_res

def find_best_match(references, queries, tolerance, key):
    """
    Calculate cosine‐greedy scores and return the best match info,
    or None for everything if no references or on error.
    """
    if not references:
        return None, None, None, None

    try:
        scores = matchms.calculate_scores(
            references=references,
            queries=queries,
            similarity_function=CosineGreedy(tolerance=tolerance),
            is_symmetric=False
        )

        best_scores, num_match_peaks, best_refs, best_queries = [], [], [], []
        for query in queries:
            matches = scores.scores_by_query(query, 'CosineGreedy_score', sort=True)
            for ref, (score, n_peaks) in matches:
                best_scores.append(score)
                num_match_peaks.append(n_peaks)
                best_refs.append(ref.metadata['peak_comments'])
                best_queries.append(query.metadata['peak_comments'])

        if not best_scores:
            return None, None, None, None
        
        #filter mathces with num_match_peaks <2 and find the maximum score among the rest
        filtered_scores = [(s, n, r, q) for s, n, r, q in zip(best_scores, num_match_peaks, best_refs, best_queries) if n >=2]
        if not filtered_scores:
            return None, None, None, None
        best_scores, num_match_peaks, best_refs, best_queries = zip(*filtered_scores)
        idx = best_scores.index(max(best_scores))
        return (
            best_scores[idx],
            num_match_peaks[idx],
            best_refs[idx],
            best_queries[idx],
        )

    except (IndexError, ValueError):
        # print(f'No match found for compound {key}')
        return None, None, None, None

In [1]:
!pip install -e "D:\NTA_analysis\LCMS_data_processing_utils[dev]"

Obtaining file:///D:/NTA_analysis/LCMS_data_processing_utils
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lcms-data-processing-utils (pyproject.toml): started
  Building editable for lcms-data-processing-utils (pyproject.toml): finished with status 'done'
  Created wheel for lcms-data-processing-utils: filename=lcms_data_processing_utils-0.1.0-0.editable-py3-none-any.whl size=4159 sha256=1dd5637938f95d98d02a771fc5ede6b9853b63633a112ebf1cc09b862ac045eb
  Stored in directory: C:\Users\yangj\AppData\Local\Temp\pip-ephem-wh


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
##restart the kernel and import my_package
import my_package

In [13]:
#load database from pickle file
from my_package.spectrum_utils import parse_mona_database

import pickle
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mbank_data_neg.pkl', 'rb') as f:
    mbank_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_data_neg.pkl', 'rb') as f:
    mona_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_obtrap_data_neg.pkl', 'rb') as f:
    mona_obtrap_data_neg = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mbank_data_pos.pkl', 'rb') as f:
    mbank_data_pos = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_data_pos.pkl', 'rb') as f:
    mona_data_pos = pickle.load(f)
with open('D:/UCSF_postdoc_topic/all_spectral_library and standards/spectral_library/mona_obtrap_data_pos.pkl', 'rb') as f:
    mona_obtrap_data_pos = pickle.load(f)

#loading inhouse library
inhouse_library_neg= r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\ENTACT_mix_standards\ENTACT_neg_spectra.msp"
inhouse_library_pos= r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\ENTACT_mix_standards\ENTACT_pos_spectra.msp"
inhouse_lib2 = r"D:\UCSF_postdoc_topic\all_spectral_library and standards\In_house_library\new_mixture_spectra.msp"

inhouse_data_neg = parse_mona_database(inhouse_library_neg, polarity="negative")
inhouse_data_pos = parse_mona_database(inhouse_library_pos, polarity="positive")
inhouse_data_neg2 = parse_mona_database(inhouse_lib2, polarity="negative")
inhouse_data_pos2 = parse_mona_database(inhouse_lib2, polarity="positive")

In [ ]:
#import mzml files for spectrum matching
manualcheck_neg = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\NegPool_11172025\neg_msms_method_manualcheck.csv'
manulacheck_pos = r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\PosPool_12012025\pos_msms_method_manualcheck.csv'
neg_check = pd.read_csv(manualcheck_neg)
pos_check = pd.read_csv(manulacheck_pos)
matches_neg = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\neg_matched_peak for ms2_with_maxsampleindex_priority.csv')
matches_pos = pd.read_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\pos_matched_peak for ms2_with_maxsampleindex_priority.csv')

import tabulate
#mutate feature_id column by combining average rt and average mass
matches_neg['feature_id'] = matches_neg['Average Rt(min)'].astype(str) + "_" + matches_neg['Average Mz'].astype(str)
matches_pos['feature_id'] = matches_pos['Average Rt(min)'].astype(str) + "_" + matches_pos['Average Mz'].astype(str)
firstmatch_neg = matches_neg[matches_neg['feature_id'].isin(neg_check['feature_id'])]
firstmatch_pos = matches_pos[matches_pos['feature_id'].isin(pos_check['feature_id'])]
#concate the two dataframes, to acquire msfile path, RT, and inchikey for potential candidates
firstmatch_neg = pd.merge(firstmatch_neg, neg_check[['feature_id', 'MS2_filepath']], on='feature_id', how='left')
firstmatch_pos = pd.merge(firstmatch_pos, pos_check[['feature_id', 'MS2_filepath']], on='feature_id', how='left')


+----+--------------+-------------------+--------------+-----------------+---------------+-----------------------------+-------------------------------------------------+------------------------------+---------------------------+-------------------+-----------+-----------------------------+-----------------------------+-----------------------------+-------------------+----------------+---------------------+-------------+---------------+----------+----------+------------------+------------------------+------------------+-------------------------------+------------------------+-------------------------+----------------------------------------------------------------------------------------------------------------------+
|    |   Unnamed: 0 |   Average Rt(min) |   Average Mz | feature_id      | DTXSID_x      | SMILES_STD                  | PREFERRED_NAME_x                                | MOLECULAR_FORMULA_original   | Pred. Ionization source   |   BloodPaperCount | 2019 PV   | InChiKey_o

In [ ]:
print(matches_neg.shape)
print(matches_pos.shape)
print(matches_neg['priority_confirmation'].value_counts())
print(matches_pos['priority_confirmation'].value_counts())

(27355, 27)
(36300, 27)
priority_confirmation
4    20070
3     7052
1      233
Name: count, dtype: int64
priority_confirmation
4    23769
3     7608
2     4651
1      272
Name: count, dtype: int64


In [ ]:
#import filtered chemical list
massspec_neg = pd.read_csv("D:/UCSF_postdoc_topic/REVEAL_first_200/chemical_list/toxtarget_with_spectrum_neg.csv")
massspec_pos = pd.read_csv("D:/UCSF_postdoc_topic/REVEAL_first_200/chemical_list/toxtarget_with_spectrum_pos.csv")

#fitler firstbatch_neg with massspec_neg by INCHIKEY
firstbatch_neg = firstmatch_neg[firstmatch_neg['InChiKey_origin'].isin(massspec_neg['INCHIKEY'])]  
print(f"Number of features in firstbatch_neg after filtering with massspec_neg: {len(firstbatch_neg)}")

firstbatch_pos = firstmatch_pos[firstmatch_pos['InChiKey_origin'].isin(massspec_pos['INCHIKEY'])]
print(f'number of chemical in firstmatch_pos after filtering with masspec_pos:{len(firstbatch_pos)}')

Number of features in firstbatch_neg after filtering with massspec_neg: 20
number of chemical in firstmatch_pos after filtering with masspec_pos:153


In [ ]:
#clean up the script and make filter on the number of matched peaks
##set threshold for peak matching
match_tolerance = 0.005 #dalton
peak_int_tol = 5 #ppm
peak_norm_tol = 5 #%
##import library for spectrum comparison
#perform spectrum search for each row by InChiKey_origin column, and add number of found spectra to the dataframe from each database.
#add tqdm progress bar to the loop

# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)
##set threshold for peak matching
match_tolerance = 0.01 #dalton
precursor_mass_tol = 10 #ppm
peak_norm_tol = 5 #%
##import library for spectrum comparison
#perform spectrum search for each row by InChiKey_origin column, and add number of found spectra to the dataframe from each database.
#add tqdm progress bar to the loop`

# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)

for iter, row in firstmatch_neg.iterrows():
    key = row['InChiKey_origin']

    #extract the query spectrum from targeted injection
    query_spectrums =[]
    target_mz = row['Average Mz']
    q_filepath = row['MS2_filepath']

    celevel = [10,20,40]
    for ce in celevel:
        ms2_query = preprocess_spectra(q_filepath, target_mz, ce, tolerance=precursor_mass_tol)
        if ms2_query is None:
            # print(f'No spectra found for compound {key} with CE {ce}')
            continue
        sspectrum = Spectrum(mz = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[0],
                             intensities = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[1], metadata = {"inchikey": key, 'peak_comments':  str(ce)+'eV'})
        query_spectrums.append(sspectrum)

    if len(query_spectrums) ==0:
        print(f'No query spectrum found for compound {key}')
        continue

    #extract the reference spectrum from database
    reference_spectrums_library = []
    reference_spectrums_insilico =[]

    sources = [
    (mbank_data_neg, reference_spectrums_library,  'mbank'),
    (mona_data_neg,  reference_spectrums_library,   'mona'),
    (mona_obtrap_data_neg, reference_spectrums_library, 'mona_obtrap'),
    (inhouse_data_neg, reference_spectrums_library, 'inhouse1'),
    (inhouse_data_neg2, reference_spectrums_library, 'inhouse2')]


    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
    reference_spectrums_library,
    query_spectrums,
    match_tolerance,
    key)
    
    firstmatch_neg.loc[iter, 'library_best_match'] = best_spectrum_library
    firstmatch_neg.loc[iter, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    firstmatch_neg.loc[iter, 'library_best_match_query'] = best_match_query_library
    firstmatch_neg.loc[iter, 'library_best_match_score_value'] = best_score_library

In [10]:
firstmatch_neg['library_best_match_score_value'].fillna(0, inplace=True)
# print(tabulate.tabulate(firstmatch_neg.head(10), headers='keys', tablefmt='psql'))

#get the summary of the library_best_match_score_value column
summary_neg = firstmatch_neg['library_best_match_score_value'].describe()
print("Summary of library_best_match_score_value:")
print(summary_neg)

Summary of library_best_match_score_value:
count    32.000000
mean      0.463953
std       0.450291
min       0.000000
25%       0.000000
50%       0.462995
75%       0.933755
max       0.999758
Name: library_best_match_score_value, dtype: float64


In [88]:
#filter rows with library_best_match_num_matchpeak >= 3, 
firstmatch_neg_filter = firstmatch_neg[firstmatch_neg['library_best_match_num_matchpeak']>=3]
print(tabulate.tabulate(firstmatch_neg_filter, headers='key', tablefmt= 'psql'))

+----+--------+-------+---------+-----------------+---------------+-----------------------------+-----------------------+----------+-----+------+-----+-----------------------------+-----------------------------+-----------------------------+-------+---------+-----+-------------+------+---------+---------+-----------------+---------+------------------+------------------+------------------+----+----------------------------------------------------------------------------------------------------------------------+--------------------------+----+------+----------+-----------------------------+
|    |        |       |         |                 |               |                             |                       |          |     |      |     |                             |                             |                             |       |         |     |             |      |         |         |                 |         |                  |                  |                  |    |        

In [ ]:
# suppress all matchms warnings
logging.getLogger("matchms").setLevel(logging.ERROR)

for iter, row in firstmatch_pos.iterrows():
    key = row['InChiKey_origin']

    #extract the query spectrum from targeted injection
    query_spectrums =[]
    target_mz = row['Average Mz']
    q_filepath = row['MS2_filepath']

    celevel = [10,20,40]
    for ce in celevel:
        ms2_query = preprocess_spectra(q_filepath, target_mz, ce, tolerance=precursor_mass_tol)
        if ms2_query is None:
            # print(f'No spectra found for compound {key} with CE {ce}')
            continue
        sspectrum = Spectrum(mz = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[0],
                             intensities = normalize_and_filter(ms2_query[0], baseline=peak_norm_tol)[1], metadata = {"inchikey": key, 'peak_comments':  str(ce)+'eV'})
        query_spectrums.append(sspectrum)

    if len(query_spectrums) ==0:
        print(f'No query spectrum found for compound {key}')
        continue

    #extract the reference spectrum from database
    reference_spectrums_library = []
    reference_spectrums_insilico =[]

    sources = [
    (mbank_data_pos, reference_spectrums_library,  'mbank'),
    (mona_data_pos,  reference_spectrums_library,   'mona'),
    (mona_obtrap_data_pos, reference_spectrums_library, 'mona_obtrap'),
    (inhouse_data_pos, reference_spectrums_library, 'inhouse1'),
    (inhouse_data_pos2, reference_spectrums_library, 'inhouse2')]

    for data, library, prefix in sources:
        spectra_dict = get_spectrum_by_key(data, key)
        for ce, spectra in spectra_dict.items():
            for sspectrum in spectra:
                mz, intensities = normalize_and_filter(sspectrum, baseline=peak_norm_tol)
                library.append(
                    Spectrum(
                        mz=mz,
                        intensities=intensities,
                        metadata={
                            'inchikey': key,
                            'peak_comments': f"{prefix}_{ce}"
                        }
                    )
                )

    best_score_library, best_num_matchpeak_library, best_spectrum_library, best_match_query_library = find_best_match(
    reference_spectrums_library,
    query_spectrums,
    match_tolerance,
    key)

    # best_score_insilico, best_num_matchpeak_insilico, best_spectrum_insilico, best_match_query_insilico = find_best_match(
    #     reference_spectrums_insilico,
    #     query_spectrums,
    #     match_tolerance,
    #     key)
    
    firstmatch_pos.loc[iter, 'library_best_match'] = best_spectrum_library
    firstmatch_pos.loc[iter, 'library_best_match_num_matchpeak'] = best_num_matchpeak_library
    firstmatch_pos.loc[iter, 'library_best_match_query'] = best_match_query_library
    firstmatch_pos.loc[iter, 'library_best_match_score_value'] = best_score_library

In [ ]:
#output matching results from the first match with targeted msms
postarget_copy = firstmatch_pos.copy()
negtarget_copy = firstmatch_neg.copy()

postarget_copy['matching_by'] = 'targetedmsms'
negtarget_copy['matching_by'] = 'targetedmsms'
# postarget_copy.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\positive_matches_targetedmsms_from12012025.csv')
# negtarget_copy.to_csv(r'D:\UCSF_postdoc_topic\REVEAL_first_400\Poolmsms_from_first400batch\analysis_results_from_20251104\negative_matches_targetedmsms_from11172025.csv')

#identification from targeted msms


In [20]:
def remove_redundant_rows4(df):
    df = df.copy()

    # Ensure numeric columns
    df["library_best_match_score_value"] = pd.to_numeric(
        df.get("library_best_match_score_value", np.nan), errors="coerce"
    )
    if "max_intensity" in df.columns:
        df["max_intensity"] = pd.to_numeric(df["max_intensity"], errors="coerce")

    kept_rows = []

    for dtxsid, sub in df.groupby("DTXSID_x"):
        sub = sub.copy()
        used = set()
        idx_list = list(sub.index)

        for i in idx_list:
            if i in used:
                continue

            row = sub.loc[i]

            # ppm and RT differences vs all rows in this DTXSID group
            ppm = abs(row["Average Mz"] - sub["Average Mz"]) / row["Average Mz"] * 1e6
            rtfilter = abs(row["Average Rt(min)"] - sub["Average Rt(min)"])

            # cluster: rows within ppm/RT thresholds
            cluster_idx = sub.index[(ppm <= 10) & (rtfilter <= 0.6)].tolist()
            cluster = sub.loc[cluster_idx]

            # choose best row in this cluster
            if cluster["library_best_match_score_value"].notna().any():
                best_idx = cluster["library_best_match_score_value"].idxmax()
            elif "max_intensity" in cluster.columns and cluster["max_intensity"].notna().any():
                best_idx = cluster["max_intensity"].idxmax()
            else:
                best_idx = i  # fallback

            kept_rows.append(df.loc[best_idx])
            used.update(cluster_idx)

    filtered_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    return filtered_df

Tmsmsmatch_pos_level1 = remove_redundant_rows4(postarget_copy_filter[postarget_copy_filter['RT_diff'] <=0.5])
Tmsmsmatch_neg_level1 = remove_redundant_rows4(negtarget_copy_filter[negtarget_copy_filter['RT_diff'] <=0.5])

Tmsmsmatch_pos_level2 = remove_redundant_rows4(postarget_copy_filter[postarget_copy_filter['RT_pos'].isna()])
Tmsmsmatch_neg_level2 = remove_redundant_rows4(negtarget_copy_filter[negtarget_copy_filter['RT_neg'].isna()])

#print match results
print(Tmsmsmatch_pos_level1.shape)
print(Tmsmsmatch_pos_level2.shape)
print(Tmsmsmatch_neg_level1.shape)
print(Tmsmsmatch_neg_level2.shape)

(1, 33)
(27, 33)
(2, 33)
(4, 33)
